In [1]:
import wandb
from wandb.apis.public.runs import Run
import math
from typing import Any
import numpy as np
import pandas as pd

In [2]:
api = wandb.Api()

In [3]:
runs = api.runs(
    "graphcore/llama-mobile",
    filters={"config.name": "test-30-12-25", "state": "finished"},
)

In [4]:
def cleanup_run(run: Run) -> dict[str, Any] | None:
    d = {}
    d["n_steps"] = run.config["training"]["n_steps"]
    d["lr_exponent"] = math.log2(run.config["training"]["optimiser"]["lr"])
    d["vqa"] = run.summary["downstream.vqa.accuracy"]
    d["ai2d"] = run.summary["downstream.ai2d.accuracy"]
    d["chartqa"] = run.summary["downstream.chartqa.accuracy"]
    d["docvqa"] = run.summary["downstream.docvqa.anls"]

    history = run.history()
    d["train_loss"] = (
        np.mean(history["train/loss"].tail(10).values)
        if "train/loss" in history
        else None
    )
    d["val_loss"] = run.summary["val/loss"] if "val/loss" in run.summary else None

    return d

In [5]:
df = pd.DataFrame([cleanup_run(run) for run in runs])

In [6]:
df.sort_values(by=["lr_exponent"], ascending=True, inplace=True)

In [7]:
# Groups lr=-16 and lr=-15 should be identical
df

,n_steps,lr_exponent,vqa,ai2d,chartqa,docvqa,train_loss,val_loss
0,256,-16.0,0.588542,0.183594,0.305664,0.227840,0.081106,0.154833
1,256,-16.0,0.639323,0.126953,0.259766,0.481211,0.082757,0.148660
2,256,-16.0,0.617188,0.415039,0.325195,0.417278,0.079369,0.163521
3,256,-16.0,0.604492,0.124023,0.257812,0.304241,0.080588,0.134035
4,256,-15.0,0.600260,0.365234,0.288086,0.461852,0.092465,0.152546
5,256,-15.0,0.541341,0.334961,0.256836,0.190062,0.092090,0.167965
6,256,-15.0,0.612956,0.126953,0.287109,0.389289,0.092310,0.141887
7,256,-15.0,0.593099,0.283203,0.281250,0.161046,0.091243,0.159891
